<a href="https://colab.research.google.com/github/PankratievaMasha/Maria/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- `Ships_an_shipbuilding_companies.csv` — данные о судостроительных компаниях, построенных судах и их классификации по размерам (company, countryLabel, shipsBuilt, sizeClass, shipsInClass)

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файл в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [8]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

if not os.path.exists("Maria"):  # ИЗМЕНЕНО: имя папки
    !git clone -q https://github.com/PankratievaMasha/Maria.git    # ИЗМЕНЕНО: URL репозитория

%cd Maria  # ИЗМЕНЕНО: переход в папку Maria

print("✅ Репозиторий готов, теперь мы работаем внутри папки Maria")  # ИЗМЕНЕНО: текст сообщения

[Errno 2] No such file or directory: 'Maria # ИЗМЕНЕНО: переход в папку Maria'
/content/Maria
✅ Репозиторий готов, теперь мы работаем внутри папки Maria


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [2]:
# 📁 Переходим в папку проекта
%cd Maria

# Проверяем, где мы теперь
import os
print("✅ Текущая папка:", os.getcwd())
print("📂 Содержимое папки:", os.listdir("."))

/content/Maria
✅ Текущая папка: /content/Maria
📂 Содержимое папки: ['data', 'notebooks', 'LICENSE', '.git', 'prompts', '.gitignore', 'README.md']


In [3]:
import pandas as pd
import os

# 🔍 Сначала посмотрим, что точно лежит в папке data
print("📂 Файлы в папке data:")
print(os.listdir("data"))

# ⚠️ Теперь пробуем найти файл (возьмём имя из списка выше)
# ВНИМАНИЕ: замените имя файла на то, что увидите в списке выше!
file_path = "data/Ships_an _shipbuilding_companies.csv"

if os.path.exists(file_path):
    print(f"\n✅ Файл найден: {file_path}")
    df_ships = pd.read_csv(file_path)
    print(f"✅ Загружено строк: {len(df_ships)}")
    print(f"✅ Столбцы: {df_ships.columns.tolist()}")
    display(df_ships.head(3))
else:
    print(f"\n❌ Файл НЕ найден. Проверьте имя в списке выше!")

📂 Файлы в папке data:
['examples', 'Ships_an _shipbuilding_companies.sparql', 'Ships_an _shipbuilding_companies.csv', 'README.md', '.gitkeep']

✅ Файл найден: data/Ships_an _shipbuilding_companies.csv
✅ Загружено строк: 2960
✅ Столбцы: ['company', 'companyLabel', 'countryLabel', 'shipsBuilt', 'sizeClass', 'shipsInClass']


,company,companyLabel,countryLabel,shipsBuilt,sizeClass,shipsInClass
0,http://www.wikidata.org/entity/Q218715,Blohm + Voss,Германия,239,5_handymax (180-200м),13
1,http://www.wikidata.org/entity/Q218715,Blohm + Voss,Германия,239,4_handysize (130-180м),65
2,http://www.wikidata.org/entity/Q239763,Avondale Shipyards,США,103,2_прибрежное (25-100м),7


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `company` с URL (ссылкой на объект Wikidata) — **сохраняем его для отладки**, но переименуем в `URL`.
- Столбцы `companyLabel`, `countryLabel` содержат читаемые подписи (название компании, страна).

В этом шаге мы:
- переименуем столбец с URL Wikidata (`company` → `URL`);
- переименуем `companyLabel → company`, `countryLabel → country`;
- приведём числовые столбцы (`shipsBuilt`, `shipsInClass`) к типу `int`.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец `URL` пригодится, если нужно будет быстро перейти к оригинальной записи компании в Викиданных.

In [4]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# 1) Удаляем столбец с URL Wikidata (company)
df_ships = df_ships.drop(columns=["company"])

# 2) Переименовываем Label-столбцы в понятные имена
df_ships = df_ships.rename(columns={
    "companyLabel": "company",
    "countryLabel": "country",
})

# 3) Приводим числовые столбцы к типу int
df_ships["shipsBuilt"] = pd.to_numeric(
    df_ships["shipsBuilt"], errors="coerce"
).fillna(0).astype(int)

df_ships["shipsInClass"] = pd.to_numeric(
    df_ships["shipsInClass"], errors="coerce"
).fillna(0).astype(int)

# Проверяем результат
print("✅ Столбцы после очистки:", df_ships.columns.tolist())
print("✅ Типы данных:")
print(df_ships.dtypes)
print("\n📋 Первые 5 строк:")
display(df_ships.head(5))

✅ Столбцы после очистки: ['company', 'country', 'shipsBuilt', 'sizeClass', 'shipsInClass']
✅ Типы данных:
company         object
country         object
shipsBuilt       int64
sizeClass       object
shipsInClass     int64
dtype: object

📋 Первые 5 строк:


,company,country,shipsBuilt,sizeClass,shipsInClass
0,Blohm + Voss,Германия,239,5_handymax (180-200м),13
1,Blohm + Voss,Германия,239,4_handysize (130-180м),65
2,Avondale Shipyards,США,103,2_прибрежное (25-100м),7
3,AG Weser,Германия,116,4_handysize (130-180м),35
4,AG Weser,Германский рейх,116,4_handysize (130-180м),35


In [5]:
# 📊 Экспресс-анализ данных

print("🏭 Уникальных компаний:", df_ships['company'].nunique())
print("🌍 Уникальных стран:", df_ships['country'].nunique())
print("📏 Уникальных классов размеров:", df_ships['sizeClass'].nunique())
print(f"\n🔢 Всего записей: {len(df_ships):,}")
print(f"🚢 Суммарно построенных судов: {df_ships['shipsBuilt'].sum():,}")

# 🏆 Топ-5 стран по количеству построенных судов
print("\n🌍 Топ-5 стран по объёмам судостроения:")
top_countries = df_ships.groupby('country')['shipsBuilt'].sum().sort_values(ascending=False).head(5)
for country, count in top_countries.items():
    print(f"  • {country}: {count:,}")

# 📦 Распределение по классам размеров
print("\n📏 Суда по классам размеров:")
size_dist = df_ships.groupby('sizeClass')['shipsInClass'].sum().sort_values(ascending=False)
for size_class, count in size_dist.head(5).items():
    print(f"  • {size_class}: {count:,}")

🏭 Уникальных компаний: 1044
🌍 Уникальных стран: 75
📏 Уникальных классов размеров: 8

🔢 Всего записей: 2,960
🚢 Суммарно построенных судов: 250,666

🌍 Топ-5 стран по объёмам судостроения:
  • Япония: 47,194
  • Республика Корея: 39,421
  • Великобритания: 25,687
  • Германия: 21,155
  • Китай: 19,496

📏 Суда по классам размеров:
  • 2_прибрежное (25-100м): 10,283
  • 6_panamax (200-295м): 8,498
  • 4_handysize (130-180м): 8,435
  • 5_handymax (180-200м): 5,460
  • 3_mini-handy (100-130м): 4,483


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о судостроении:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по числовым показателям (`shipsBuilt`, `shipsInClass`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро выводить информацию о DataFrame.

> 💡 **После очистки** у нас остались столбцы:
> - `company` — название судостроительной компании
> - `country` — страна базирования
> - `shipsBuilt` — общее количество построенных судов
> - `sizeClass` — класс размера судов
> - `shipsInClass` — количество судов в данном классе

In [6]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    display(df.head(n))  # ИЗМЕНЕНО: display() лучше для таблиц в Colab

# 🔍 Шаг 3. Обзор данных

# ИЗМЕНЕНО: вызываем функцию только для df_ships
show_info(df_ships, "Данные о судостроительных компаниях (df_ships)")

# ИЗМЕНЕНО: статистика по числовым столбцам судов
print("\n📈 Статистика по числовым показателям:")
print(df_ships[["shipsBuilt", "shipsInClass"]].describe())

# 🎁 Бонус: уникальные значения категориальных столбцов
print("\n🏭 Уникальные компании (первые 10):")
print(df_ships["company"].unique()[:10])

print("\n🌍 Уникальные страны:")
print(df_ships["country"].unique())

print("\n📏 Классы размеров судов:")
print(df_ships["sizeClass"].unique())


📊 Данные о судостроительных компаниях (df_ships)
Размер: (2960, 5)
Столбцы: company, country, shipsBuilt, sizeClass, shipsInClass

Первые строки:


,company,country,shipsBuilt,sizeClass,shipsInClass
0,Blohm + Voss,Германия,239,5_handymax (180-200м),13
1,Blohm + Voss,Германия,239,4_handysize (130-180м),65
2,Avondale Shipyards,США,103,2_прибрежное (25-100м),7
3,AG Weser,Германия,116,4_handysize (130-180м),35
4,AG Weser,Германский рейх,116,4_handysize (130-180м),35



📈 Статистика по числовым показателям:
        shipsBuilt  shipsInClass
count  2960.000000   2960.000000
mean     84.684459     13.579054
std     146.121160     36.154028
min       3.000000      1.000000
25%      11.000000      2.000000
50%      39.000000      4.000000
75%     103.000000     12.000000
max    1481.000000    699.000000

🏭 Уникальные компании (первые 10):
['Blohm + Voss' 'Avondale Shipyards' 'AG Weser' 'Ольборгская судоверфь'
 'Neptun Werft' 'Samsung Heavy Industries' 'Bremer Vulkan'
 'William Beardmore and Company' 'Mitsubishi Heavy Industries'
 'Escher Wyss & Cie.']

🌍 Уникальные страны:
['Германия' 'США' 'Германский рейх' 'Дания' 'Республика Корея'
 'Великобритания' 'Япония' 'Швейцария' 'Россия' 'Нидерланды' 'Италия' nan
 'Украина' 'Польша' 'Швеция' 'Бельгия' 'Веймарская республика'
 'Германская империя' 'Австрия' 'Нацистская Германия' 'Финляндия'
 'Испания' 'Франция' 'Австралия' 'Норвегия' 'Индонезия' 'Пруссия'
 'Хорватия' 'ГДР' 'Китай' 'Румыния' 'Вольный город Данциг

## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** компаний, стран и классов размеров есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу записей);
- **какие классы размеров судов самые популярные** (Топ‑5 по числу записей);
- **распределение числовых показателей** (`shipsBuilt`, `shipsInClass`).

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_ships["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

> 💡 **Зачем это нужно:** Быстрая валидация помогает обнаружить:
> - неожиданные пропуски или дубликаты
> - ошибки в названиях стран/компаний
> - аномальные значения в числовых столбцах

In [7]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных о судостроении")

# 📊 Основной датасет: компании, страны, суда
print("\n📊 Датасет: Судостроительные компании (df_ships)")
print("Уникальных компаний:", df_ships["company"].nunique())
print("Уникальных стран:", df_ships["country"].nunique())
print("Уникальных классов размеров:", df_ships["sizeClass"].nunique())
print("Всего записей (строк):", len(df_ships))

# 🌍 Топ-5 стран по числу записей
print("\n🌍 Топ-5 стран по числу записей:")
print(df_ships["country"].value_counts().head())

# 📏 Топ-5 классов размеров судов
print("\n📏 Топ-5 классов размеров судов:")
print(df_ships["sizeClass"].value_counts().head())

# 🏭 Топ-10 компаний по числу записей
print("\n🏭 Топ-10 компаний по числу записей:")
print(df_ships["company"].value_counts().head(10))

# 🔢 Проверка числовых столбцов на аномалии
print("\n🔢 Проверка числовых показателей:")
print(f"  • shipsBuilt: мин={df_ships['shipsBuilt'].min()}, макс={df_ships['shipsBuilt'].max()}, среднее={df_ships['shipsBuilt'].mean():.1f}")
print(f"  • shipsInClass: мин={df_ships['shipsInClass'].min()}, макс={df_ships['shipsInClass'].max()}, среднее={df_ships['shipsInClass'].mean():.1f}")

# ⚠️ Проверка на пропущенные значения
print("\n❓ Пропущенные значения по столбцам:")
print(df_ships.isnull().sum())

# 🎁 Бонус: есть ли дубликаты строк?
print(f"\n🔄 Дубликаты строк: {df_ships.duplicated().sum()}")

🔍 Быстрая проверка данных о судостроении

📊 Датасет: Судостроительные компании (df_ships)
Уникальных компаний: 1044
Уникальных стран: 75
Уникальных классов размеров: 8
Всего записей (строк): 2960

🌍 Топ-5 стран по числу записей:
country
Великобритания    297
Германия          289
Япония            286
Норвегия          242
США               227
Name: count, dtype: int64

📏 Топ-5 классов размеров судов:
sizeClass
2_прибрежное (25-100м)     882
3_mini-handy (100-130м)    567
4_handysize (130-180м)     547
6_panamax (200-295м)       303
5_handymax (180-200м)      280
Name: count, dtype: int64

🏭 Топ-10 компаний по числу записей:
company
Deutsche Werft                       30
Uljanik                              21
Flender Werke                        15
3. Maj                               15
AG Vulcan Stettin                    14
Bremer Vulkan                        12
Howaldtswerke Hamburg                12
AG Weser                             12
Керченский судостроительный завод    1

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨